[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsnhiii/vaccine_tcell_model/blob/main/notebooks/cancer_tcr_model_colab.ipynb)

# Cancer / TCR-signaling model -- interactive explorer

This notebook runs the `vaccine_tcell_model` package's cancer/TCR-signaling extension (`models.cancer_tcr`), built for comparing TCR signaling strength across dosing schedules in a cancer (mouse/patient) context.

**What's in here:**
- **Part 1** -- change one dosing schedule and every model parameter (antigen affinity, kinetic-proofreading chain, T-cell growth) with sliders, and see the full pipeline (antigen through TCR signal) replot live.
- **Part 2** -- compare several dosing schedules side by side, for a chosen antigen affinity.
- **Part 3** -- a summary of what every parameter means and this model's documented caveats (units, timescale-separation assumption, what's NOT implemented yet).

Full documentation, equation-by-equation provenance, and every caveat in more depth: [`docs/cancer_tcr_model.md`](https://github.com/itsnhiii/vaccine_tcell_model/blob/main/docs/cancer_tcr_model.md) in the source repository.

In [ ]:
# Installs the package straight from GitHub -- this is the single source of truth for the
# model's equations, so this notebook never has its own out-of-sync copy of the logic.
!pip install -q "git+https://github.com/itsnhiii/vaccine_tcell_model.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox, Label
from IPython.display import display

from vaccine_tcell_model.dosing import DoseSchedule
from vaccine_tcell_model.models.cancer_tcr import simulate_cancer_tcr, compute_tcr_signal
from vaccine_tcell_model.parameters import (
    default_cancer_tcr_parameters,
    cancer_tcr_default_initial_conditions,
    default_tcr_signal_parameters,
)

SLIDER_STYLE = {"description_width": "280px"}
SLIDER_LAYOUT = widgets.Layout(width="520px")

def _style(w):
    w.style = SLIDER_STYLE
    w.layout = SLIDER_LAYOUT
    if hasattr(w, "continuous_update"):
        w.continuous_update = False
    return w

# Every distinct way to specify a DoseSchedule (dosing/schedules.py),
# each wrapped as `lambda total_antigen_dose, total_adjuvant_dose: DoseSchedule(...)`
# so all of them can be driven by the same two total-dose sliders below.
# This is the ONE place these are defined -- Part 1's dropdown and Part 2's
# multi-select both just read `list(SCHEDULE_BUILDERS)`, so adding an entry
# here makes it available in both places automatically.
SCHEDULE_BUILDERS = {
    # Convenience constructor: a single dose of the full amount at day 0.
    "Bolus (single dose, day 0)": lambda ag, adj: DoseSchedule.bolus(
        total_antigen_dose=ag, total_adjuvant_dose=adj),

    # Fractions of a total dose, antigen and adjuvant sharing the same times.
    "2-dose escalation (20% day 0, 80% day 7)": lambda ag, adj: DoseSchedule(
        times=[0, 7], antigen_fractions=[0.2, 0.8], adjuvant_fractions=[0.2, 0.8],
        total_antigen_dose=ag, total_adjuvant_dose=adj),

    # Exponential-escalation convenience constructor (matches the reference
    # Science-paper dose generator, docs/equations.md Flag S5); both
    # channels escalate together by default.
    "7-dose escalation (over 12 days, matched channels)": lambda ag, adj: DoseSchedule.from_exponential_escalation(
        numshot=7, k=1.0, duration=12, total_antigen_dose=ag, total_adjuvant_dose=adj),

    # Convenience constructor: N equal-sized doses at the given times.
    "4 equal doses (days 0, 3, 6, 9)": lambda ag, adj: DoseSchedule.equal_doses(
        times=[0, 3, 6, 9], total_antigen_dose=ag, total_adjuvant_dose=adj),

    # Absolute dose AMOUNTS directly (scaled here by the total-dose sliders
    # to keep a 0.5:1:2 antigen ratio and equal adjuvant doses; the
    # underlying DoseSchedule call itself takes plain numbers, no
    # total_dose needed -- see the cookbook section below).
    "Absolute dose ratios (0.5:1:2 antigen, equal adjuvant)": lambda ag, adj: DoseSchedule(
        times=[0, 7, 14],
        antigen_doses=[0.5 * ag, 1.0 * ag, 2.0 * ag],
        adjuvant_doses=[0.5 * adj, 0.5 * adj, 0.5 * adj]),

    # Antigen and adjuvant on COMPLETELY INDEPENDENT timelines: antigen
    # escalates over 7 doses while adjuvant is a single upfront bolus
    # (the "escalating antigen, adjuvant bolus" variant explored in
    # examples/07_cancer_tcr_dosing_and_adjuvant.py).
    "7-dose antigen escalation, adjuvant bolus (independent timing)": lambda ag, adj: DoseSchedule(
        antigen_times=[0, 2, 4, 6, 8, 10, 12],
        antigen_fractions=[0.05, 0.07, 0.09, 0.12, 0.16, 0.22, 0.29],
        adjuvant_times=[0], adjuvant_fractions=[1.0],
        total_antigen_dose=ag, total_adjuvant_dose=adj),

    # Same exponential-escalation constructor, but with the adjuvant_*
    # arguments overridden so adjuvant escalates on its OWN schedule
    # (2 doses over 7 days) instead of matching the antigen's (7 over 12).
    "7-dose antigen, 2-dose adjuvant (independent escalation)": lambda ag, adj: DoseSchedule.from_exponential_escalation(
        numshot=7, k=1.0, duration=12,
        adjuvant_numshot=2, adjuvant_k=2.0, adjuvant_duration=7,
        total_antigen_dose=ag, total_adjuvant_dose=adj),

    # Edge case: antigen given, but adjuvant totally absent (a dose of
    # 0.0, ignoring the adjuvant-dose slider). Instructive because in this
    # model adjuvant is what recruits the innate cells DCs are drawn from --
    # with none, DC/aDC_Ag/T all stay at zero for the whole simulation
    # (see docs/cancer_tcr_model.md).
    "No adjuvant at all (ignores adjuvant-dose slider)": lambda ag, adj: DoseSchedule(
        antigen_times=[0], antigen_fractions=[1.0], total_antigen_dose=ag,
        adjuvant_times=[0], adjuvant_doses=[0.0]),
}

UNITS_FOOTNOTE = (
    "Cell counts and antigen dose units are this model's own fitted, normalized scale, "
    "not independently validated absolute values."
)

## Dosing schedule cookbook -- every way to specify a `DoseSchedule`

`DoseSchedule` (`vaccine_tcell_model.dosing.schedules`) supports several distinct ways to describe a regimen. This section shows each one as runnable code, prints the resolved antigen/adjuvant dose events, and plots the resulting `Ag(t)`/`Adj(t)` traces so the syntax and the shape it produces are both visible together. Parts 1 and 2 below use only three of these (bolus, 2-dose, 7-dose escalation) for the interactive sliders -- everything here is a static reference you can copy into your own cell and adapt.

In [ ]:
# The cookbook is the same SCHEDULE_BUILDERS dict defined above (in the
# imports cell) and used by Part 1's dropdown and Part 2's multi-select --
# nothing here is a separate, duplicated definition. Calling each builder
# with a representative total dose of 1.0 shows what it resolves to.
for name, builder in SCHEDULE_BUILDERS.items():
    schedule = builder(1.0, 1.0)
    ag_doses = tuple(round(d, 3) for d in schedule.antigen_doses)
    adj_doses = tuple(round(d, 3) for d in schedule.adjuvant_doses)
    print(name)
    print(f"  antigen:  times={schedule.antigen_times}  doses={ag_doses}")
    print(f"  adjuvant: times={schedule.adjuvant_times}  doses={adj_doses}")
    print()

In [ ]:
_params = default_cancer_tcr_parameters()
_ic = cancer_tcr_default_initial_conditions(_params)
_t_eval = np.linspace(0, 21, 421)

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharex=True)
for ax, (name, builder) in zip(axes.flat, SCHEDULE_BUILDERS.items()):
    result = simulate_cancer_tcr(_params, _ic, builder(1.0, 1.0), t_end=21.0, t_eval=_t_eval)
    ax.plot(result.time, result.trajectory("Ag"), label="Antigen")
    ax.plot(result.time, result.trajectory("Adj"), label="Adjuvant", linestyle="--")
    ax.set_title(name, fontsize=8)
    ax.set_xlabel("time (days)")
    ax.set_ylim(bottom=0)
axes.flat[0].set_ylabel("dose units (linear)")
axes.flat[4].set_ylabel("dose units (linear)")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9)
fig.suptitle("Every SCHEDULE_BUILDERS entry (antigen solid, adjuvant dashed)")
fig.text(0.5, -0.02, UNITS_FOOTNOTE, ha="center", fontsize=9, style="italic", wrap=True)
fig.tight_layout(rect=(0, 0, 0.82, 1))
plt.show()

## Part 1 -- single-run explorer

Change the dosing schedule and any parameter below; the six-panel plot and the printed peak signal update automatically. Sliders under **Advanced** control the population-dynamics side of the model (T-cell growth, pMHC loading/decay) -- the defaults are this project's illustrative values (`docs/cancer_tcr_model.md`), not literature-fixed constants, so they are meant to be explored/fit, same as `K_D`/`k_p`/`N`.

In [ ]:
# -- core knobs -----------------------------------------------------------
schedule_dropdown = _style(widgets.Dropdown(
    options=list(SCHEDULE_BUILDERS), value="7-dose escalation (over 12 days, matched channels)",
    description="Dosing schedule"))
total_antigen_slider = _style(widgets.FloatLogSlider(
    value=1.0, base=10, min=-1, max=1, step=0.05, description="Total antigen dose"))
total_adjuvant_slider = _style(widgets.FloatLogSlider(
    value=1.0, base=10, min=-1, max=1, step=0.05, description="Total adjuvant dose"))
kd_slider = _style(widgets.FloatLogSlider(
    value=50.0, base=10, min=0, max=2.5, step=0.02,
    description="Antigen affinity K_D (micromolar)"))
kp_slider = _style(widgets.FloatSlider(
    value=0.3, min=0.05, max=1.0, step=0.05,
    description="Proofreading step rate k_p (per second)"))
n_slider = _style(widgets.IntSlider(
    value=3, min=2, max=4, step=1, description="Number of proofreading steps N"))
t_end_slider = _style(widgets.FloatSlider(
    value=21.0, min=7.0, max=42.0, step=1.0, description="Simulation length (days)"))

# -- advanced knobs (population dynamics) ----------------------------------
k_on_slider = _style(widgets.FloatLogSlider(
    value=0.01, base=10, min=-3, max=0, step=0.1,
    description="k_on (per micromolar per second)"))
alpha_slider = _style(widgets.FloatSlider(
    value=1.5, min=0.1, max=5.0, step=0.1,
    description="alpha, T-cell proliferation rate (per day)"))
delta_slider = _style(widgets.FloatSlider(
    value=0.22, min=0.01, max=1.0, step=0.01,
    description="delta, T-cell turnover rate (per day)"))
kt_slider = _style(widgets.FloatLogSlider(
    value=10.0, base=10, min=-1, max=3, step=0.1,
    description="K_T, T-cell growth saturation constant"))
k_load_slider = _style(widgets.FloatLogSlider(
    value=1.0, base=10, min=-2, max=2, step=0.1,
    description="k_load, pMHC loading rate (per day)"))
mu_pmhc_slider = _style(widgets.FloatLogSlider(
    value=1.0, base=10, min=-2, max=2, step=0.1,
    description="mu_pMHC, pMHC decay rate (per day)"))


def run_and_plot(schedule_name, total_antigen_dose, total_adjuvant_dose, K_D, k_p, N, t_end,
                 k_on, alpha, delta, K_T, k_load, mu_pMHC):
    params = default_cancer_tcr_parameters(k_load=k_load, mu_pMHC=mu_pMHC, K_T=K_T)
    params = params.with_value("alpha", alpha)
    params = params.with_value("delta", delta)
    ic = cancer_tcr_default_initial_conditions(params)
    schedule = SCHEDULE_BUILDERS[schedule_name](total_antigen_dose, total_adjuvant_dose)
    t_eval = np.linspace(0, t_end, int(t_end * 20) + 1)
    result = simulate_cancer_tcr(params, ic, schedule, t_end=t_end, t_eval=t_eval)

    signal_params = default_tcr_signal_parameters(K_D=K_D, k_on=k_on, k_p=k_p, N=float(N))
    signal_df = compute_tcr_signal(result, signal_params)

    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    panels = [
        ("Ag", "Antigen (dose units)"),
        ("aDC_Ag", "Number of dendritic cells"),
        ("pMHC_density", "Peptide-MHC density"),
        ("T", "Number of T cells"),
    ]
    for ax, (name, ylabel) in zip(axes.flat[:4], panels):
        ax.plot(result.time, result.trajectory(name))
        ax.set_title(name)
        ax.set_xlabel("time (days)")
        ax.set_ylabel(ylabel)
        if name == "Ag":
            # Linear 0-based scale (plain floats) rather than log/scientific
            # notation -- antigen's own dynamic range is small enough that
            # log scale just obscures it behind tiny decay-tail values.
            ax.set_ylim(bottom=0)
        else:
            ax.set_yscale("log")

    axes.flat[4].plot(signal_df["time"], signal_df["S_contact"], color="C2")
    axes.flat[4].set_title("S_contact")
    axes.flat[4].set_xlabel("time (days)")
    axes.flat[4].set_ylabel("TCR signal per cell")

    axes.flat[5].plot(signal_df["time"], signal_df["S_pop"], color="C3")
    axes.flat[5].set_title("S_pop")
    axes.flat[5].set_xlabel("time (days)")
    axes.flat[5].set_ylabel("TCR signal, population")

    fig.suptitle(f"{schedule_name} -- K_D = {K_D:.3g} micromolar, k_p = {k_p:.3g}/s, N = {N}")
    fig.text(0.5, -0.02, UNITS_FOOTNOTE, ha="center", fontsize=9, style="italic", wrap=True)
    fig.tight_layout()
    plt.show()

    peak_row = signal_df.loc[signal_df["S_pop"].idxmax()]
    print(f"Peak population TCR signal: {peak_row['S_pop']:.4g} (at day {peak_row['time']:.2f})")
    print(f"Peak per-cell TCR signal:   {signal_df['S_contact'].max():.4g} (dimensionless, 0 to 1)")


core_controls = VBox([
    Label("Dosing schedule"), schedule_dropdown, total_antigen_slider, total_adjuvant_slider,
    Label("TCR-signaling kinetics"), kd_slider, kp_slider, n_slider, t_end_slider,
])
advanced_controls = VBox([
    Label("Advanced: population-dynamics parameters"),
    k_on_slider, alpha_slider, delta_slider, kt_slider, k_load_slider, mu_pmhc_slider,
])

part1_out = interactive_output(run_and_plot, {
    "schedule_name": schedule_dropdown,
    "total_antigen_dose": total_antigen_slider,
    "total_adjuvant_dose": total_adjuvant_slider,
    "K_D": kd_slider,
    "k_p": kp_slider,
    "N": n_slider,
    "t_end": t_end_slider,
    "k_on": k_on_slider,
    "alpha": alpha_slider,
    "delta": delta_slider,
    "K_T": kt_slider,
    "k_load": k_load_slider,
    "mu_pMHC": mu_pmhc_slider,
})

display(HBox([VBox([core_controls, advanced_controls]), ]))
display(part1_out)

## Part 2 -- compare dosing schedules side by side

Pick which schedules to overlay (Ctrl/Cmd-click for multiple), a total dose, and an antigen affinity, and see the same six-panel pipeline as Part 1 (antigen, dendritic cells, peptide-MHC density, T cells, per-cell signal, population signal), all overlaid by schedule. This is the model's core intended use: comparing candidate dosing schedules under the same assumed antigen affinity.

In [ ]:
schedule_multiselect = _style(widgets.SelectMultiple(
    options=list(SCHEDULE_BUILDERS), value=tuple(SCHEDULE_BUILDERS),
    description="Schedules to compare", rows=8))
total_antigen_slider2 = _style(widgets.FloatLogSlider(
    value=1.0, base=10, min=-1, max=1, step=0.05, description="Total antigen dose"))
total_adjuvant_slider2 = _style(widgets.FloatLogSlider(
    value=1.0, base=10, min=-1, max=1, step=0.05, description="Total adjuvant dose"))
kd_slider2 = _style(widgets.FloatLogSlider(
    value=50.0, base=10, min=0, max=2.5, step=0.02,
    description="Antigen affinity K_D (micromolar)"))
t_end_slider2 = _style(widgets.FloatSlider(
    value=21.0, min=7.0, max=42.0, step=1.0, description="Simulation length (days)"))


def compare_schedules(selected_schedules, total_antigen_dose, total_adjuvant_dose, K_D, t_end):
    if not selected_schedules:
        print("Select at least one schedule to compare.")
        return

    params = default_cancer_tcr_parameters()
    ic = cancer_tcr_default_initial_conditions(params)
    signal_params = default_tcr_signal_parameters(K_D=K_D)
    t_eval = np.linspace(0, t_end, int(t_end * 20) + 1)

    # Same six-panel pipeline as Part 1, but one line per selected
    # schedule in every panel instead of a single run.
    panels = [
        ("Ag", "Antigen (dose units)", False),
        ("aDC_Ag", "Number of dendritic cells", True),
        ("pMHC_density", "Peptide-MHC density", True),
        ("T", "Number of T cells", True),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    peak_signal = {}
    for name in selected_schedules:
        schedule = SCHEDULE_BUILDERS[name](total_antigen_dose, total_adjuvant_dose)
        result = simulate_cancer_tcr(params, ic, schedule, t_end=t_end, t_eval=t_eval)
        signal_df = compute_tcr_signal(result, signal_params)

        for ax, (state_name, _, _) in zip(axes.flat[:4], panels):
            ax.plot(result.time, result.trajectory(state_name), label=name)
        axes.flat[4].plot(signal_df["time"], signal_df["S_contact"], label=name)
        axes.flat[5].plot(signal_df["time"], signal_df["S_pop"], label=name)
        peak_signal[name] = signal_df["S_pop"].max()

    for ax, (state_name, ylabel, log_scale) in zip(axes.flat[:4], panels):
        ax.set_title(state_name)
        ax.set_xlabel("time (days)")
        ax.set_ylabel(ylabel)
        if log_scale:
            ax.set_yscale("log")
        else:
            # Ag: linear 0-based scale (plain floats), not log/scientific notation.
            ax.set_ylim(bottom=0)

    axes.flat[4].set_title("S_contact")
    axes.flat[4].set_xlabel("time (days)")
    axes.flat[4].set_ylabel("TCR signal per cell")

    axes.flat[5].set_title("S_pop")
    axes.flat[5].set_xlabel("time (days)")
    axes.flat[5].set_ylabel("TCR signal, population")

    # One shared legend outside every axes -- all six panels use the
    # same color-to-schedule mapping, so a single legend covers all of them
    # without covering any data line.
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9)
    fig.suptitle(f"Dosing schedule comparison (K_D = {K_D:.3g} micromolar)")
    fig.text(0.5, -0.03, UNITS_FOOTNOTE, ha="center", fontsize=9, style="italic", wrap=True)
    fig.tight_layout(rect=(0, 0, 0.82, 1))
    plt.show()

    print("Peak population TCR signal by schedule:")
    for name, value in sorted(peak_signal.items(), key=lambda kv: -kv[1]):
        print(f"  {name}: {value:.4g}")


part2_controls = VBox([
    schedule_multiselect, total_antigen_slider2, total_adjuvant_slider2, kd_slider2, t_end_slider2,
])
part2_out = interactive_output(compare_schedules, {
    "selected_schedules": schedule_multiselect,
    "total_antigen_dose": total_antigen_slider2,
    "total_adjuvant_dose": total_adjuvant_slider2,
    "K_D": kd_slider2,
    "t_end": t_end_slider2,
})

display(part2_controls)
display(part2_out)

## Part 3 -- what the parameters mean, and this model's caveats

**Dosing / population parameters:**
- `Total antigen dose`, `Total adjuvant dose` -- normalized dose units (default schedules use `1.0` as "the full dose"); not a physical concentration.
- `alpha` -- maximum T-cell proliferation rate. `delta` -- T-cell homeostatic turnover rate (pulls the population back toward its baseline `T0` in the absence of antigen; NOT a plain unconditional death rate -- see `models/cancer_tcr/tcell.py`).
- `K_T` -- T-cell growth saturation constant. **Not the same thing as `K_D`** -- `K_T` shapes how fast the T-cell *population* grows, `K_D` shapes how strongly a single engaged TCR *signals*. Keeping them separate is deliberate.
- `k_load`, `mu_pMHC` -- how fast peptide-MHC density builds up from activated, antigen-loaded dendritic cells, and how fast it decays.

**TCR-signaling parameters (the Chakraborty & Weiss 2014 kinetic-proofreading layer):**
- `K_D` -- antigen affinity (dissociation constant), in micromolar. **This is the model's required scientific input** -- the illustrative default is not a real antigen's measured affinity.
- `k_on` -- TCR-pMHC association rate, held fixed (both source papers find it varies little across peptides; the affinity difference is carried by the off-rate instead).
- `k_p`, `N` -- the kinetic-proofreading chain's forward step rate and number of steps. Literature-anchored defaults (~2-4 steps, ~seconds each), but genuinely fittable against real experimental TCR-signal data, not fixed constants.

**Caveats worth remembering before trusting absolute numbers:**
1. Dendritic cell and T-cell counts, and antigen dose units, are this model's own fitted, normalized scale -- trust relative comparisons (across schedules/affinities), not absolute magnitudes, until calibrated against real data.
2. The kinetic-proofreading calculation is a closed-form, quasi-steady-state approximation, justified because it completes in seconds while this model's population dynamics evolve over days -- a large timescale separation, but still an approximation.
3. A serial-triggering / "optimal dwell time" correction is deliberately NOT implemented: Chakraborty & Weiss 2014 report no clear experimental support for it. Setting `include_serial_triggering` raises `NotImplementedError` rather than silently doing nothing.

Full detail on every point above: [`docs/cancer_tcr_model.md`](https://github.com/itsnhiii/vaccine_tcell_model/blob/main/docs/cancer_tcr_model.md).